# SAM3 image seed
Private Kaggle GPU notebook. Attach exactly one sam3.pt checkpoint and one private payload.

In [ ]:
"""Kaggle GPU script: detect editable CVAT frame-0 boxes using SAM3 text prompts."""

from __future__ import annotations

import hashlib
import json
import math
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
MAX_BOXES = 80
MIN_SCORE = 0.55


def _setup() -> None:
    if sys.version_info < (3, 12) or not torch.cuda.is_available():
        raise RuntimeError("SAM3 frame-0 labeling requires Kaggle Python 3.12+ and a CUDA GPU")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         "git+https://github.com/facebookresearch/sam3.git",
         "timm>=1.0.17", "ftfy==6.1.1", "iopath>=0.1.10", "portalocker", "huggingface_hub"],
        check=True,
    )


def _checkpoint() -> Path:
    candidates = list(INPUT_ROOT.glob("**/sam3.pt"))
    if len(candidates) != 1 or candidates[0].stat().st_size < 1_000_000:
        raise RuntimeError("Expected one sam3.pt in the private checkpoint dataset")
    return candidates[0]


def _inputs() -> tuple[dict, Path]:
    archives = list(INPUT_ROOT.glob("**/autolabel-sam3-seed-*/payload.zip"))
    extracted = list(INPUT_ROOT.glob("**/autolabel-sam3-seed-*/request.json"))
    if len(archives) + len(extracted) != 1:
        raise RuntimeError("Expected exactly one private AutoLabel SAM3 seed input")
    if extracted:
        target = extracted[0].parent
    else:
        target = WORK_ROOT / "autolabel-seed-input"
        target.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            if set(archive.namelist()) != {"request.json", "frame.jpg"}:
                raise RuntimeError("Unexpected SAM3 seed input files")
            archive.extractall(target)
    request = json.loads((target / "request.json").read_text(encoding="utf-8"))
    if hashlib.sha256((target / "frame.jpg").read_bytes()).hexdigest() != request["frame_sha256"]:
        raise RuntimeError("SAM3 seed frame does not match request")
    return request, target / "frame.jpg"


def _iou(left: dict, right: dict) -> float:
    a, b = left["bbox"], right["bbox"]
    width = max(0.0, min(a["xbr"], b["xbr"]) - max(a["xtl"], b["xtl"]))
    height = max(0.0, min(a["ybr"], b["ybr"]) - max(a["ytl"], b["ytl"]))
    intersection = width * height
    area_a = (a["xbr"] - a["xtl"]) * (a["ybr"] - a["ytl"])
    area_b = (b["xbr"] - b["xtl"]) * (b["ybr"] - b["ytl"])
    return intersection / max(area_a + area_b - intersection, 1e-9)


def _detect(request: dict, image_path: Path, checkpoint: Path,
            runtime: dict | None = None) -> list[dict]:
    model_import_started = time.perf_counter()
    from sam3.model.sam3_image_processor import Sam3Processor
    from sam3.model_builder import build_sam3_image_model

    from kaggle.runtime_probe import begin_cuda, duration_stats, end_cuda

    if runtime is not None:
        runtime["timing_s"]["model_import"] = time.perf_counter() - model_import_started
    image = Image.open(image_path).convert("RGB")
    if image.size != (request["width"], request["height"]):
        raise RuntimeError("SAM3 seed image size does not match request")
    begin_cuda(torch)
    load_started = time.perf_counter()
    model = build_sam3_image_model(checkpoint_path=str(checkpoint), load_from_HF=False)
    if runtime is not None:
        runtime["peak_cuda_allocated_bytes"]["model_load"] = end_cuda(torch)
        runtime["timing_s"]["model_load"] = time.perf_counter() - load_started
    processor = Sam3Processor(model, confidence_threshold=MIN_SCORE)
    begin_cuda(torch)
    image_started = time.perf_counter()
    state = processor.set_image(image)
    if runtime is not None:
        runtime["peak_cuda_allocated_bytes"]["image_encode"] = end_cuda(torch)
        runtime["timing_s"]["image_encode"] = time.perf_counter() - image_started
    candidates = []
    prompt_times: list[float] = []
    prompt_peaks: list[int | None] = []
    for label in request["labels"]:
        prompt = label["prompt"].replace("_", " ").strip()
        if not prompt:
            continue
        processor.reset_all_prompts(state)
        begin_cuda(torch)
        prompt_started = time.perf_counter()
        with torch.inference_mode():
            output = processor.set_text_prompt(state=state, prompt=prompt)
        prompt_peaks.append(end_cuda(torch))
        prompt_times.append(time.perf_counter() - prompt_started)
        for raw_box, raw_score in zip(output["boxes"].tolist(), output["scores"].tolist(), strict=True):
            coordinates = [float(value) for value in raw_box]
            score = float(raw_score)
            if not all(math.isfinite(value) for value in (*coordinates, score)) or score < MIN_SCORE:
                continue
            x1, y1, x2, y2 = coordinates
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(image.width), x2), min(float(image.height), y2)
            if x2 - x1 < 4 or y2 - y1 < 4:
                continue
            candidates.append({
                "label_id": label["id"], "score": score,
                "bbox": {"xtl": x1, "ytl": y1, "xbr": x2, "ybr": y2},
            })
    selected: list[dict] = []
    for candidate in sorted(candidates, key=lambda item: item["score"], reverse=True):
        if all(_iou(candidate, previous) < 0.8 for previous in selected):
            selected.append(candidate)
        if len(selected) == MAX_BOXES:
            break
    if runtime is not None:
        runtime["timing_s"]["inference_prompt"] = duration_stats(prompt_times)
        runtime["peak_cuda_allocated_bytes"]["inference_prompts"] = prompt_peaks
    return selected


def main() -> None:
    from kaggle.runtime_probe import environment, sha256_file

    input_started = time.perf_counter()
    request, image_path = _inputs()
    input_seconds = time.perf_counter() - input_started
    setup_started = time.perf_counter()
    _setup()
    runtime = environment(torch, include_sam3=True)
    runtime["timing_s"]["input_validation"] = input_seconds
    runtime["timing_s"]["dependency_setup"] = time.perf_counter() - setup_started
    device = torch.cuda.get_device_properties(0)
    dtype = torch.bfloat16 if device.major >= 8 else torch.float16
    checkpoint = _checkpoint()
    runtime["checkpoint_sha256"] = sha256_file(checkpoint)
    runtime["dtype"] = str(dtype)
    runtime["autocast"] = {"enabled": True, "device": "cuda", "dtype": str(dtype)}
    runtime["timing_s"]["checkpoint_download"] = None  # Kaggle mounts the dataset before script start.
    with torch.autocast("cuda", dtype=dtype):
        detections = _detect(request, image_path, checkpoint, runtime=runtime)
    result = {"stage": "sam3-seed", "job_id": request["job_id"],
              "detections": detections, "runtime": runtime}
    (WORK_ROOT / "result.json").write_text(json.dumps(result, separators=(",", ":")), encoding="utf-8")



In [ ]:
import sys, types
_package = types.ModuleType('kaggle')
sys.modules.setdefault('kaggle', _package)
_module = types.ModuleType('kaggle.runtime_probe')
exec('"""Optional, JSON-safe GPU runtime observations for finite model jobs."""\n\nimport hashlib\nimport importlib.metadata\nimport json\nimport statistics\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n\ndef _version(package):\n    try:\n        return importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError:\n        return None\n\n\ndef sam3_commit():\n    """Read the installed VCS commit when pip preserved it; never invent a revision."""\n    try:\n        distribution = importlib.metadata.distribution("sam3")\n        direct_url = distribution.read_text("direct_url.json")\n        if direct_url:\n            commit = json.loads(direct_url).get("vcs_info", {}).get("commit_id")\n            if commit:\n                return commit\n    except (importlib.metadata.PackageNotFoundError, ValueError, TypeError):\n        pass\n    try:\n        import sam3\n\n        root = Path(sam3.__file__).resolve().parent\n        result = subprocess.run(\n            ["git", "-C", str(root), "rev-parse", "HEAD"],\n            capture_output=True, text=True, timeout=5, check=False,\n        )\n        return result.stdout.strip() if result.returncode == 0 else None\n    except (ImportError, OSError, subprocess.TimeoutExpired, TypeError):\n        return None\n\n\ndef environment(torch, *, include_sam3=False):\n    versions = {\n        "python": sys.version.split()[0],\n        "torch": torch.__version__,\n        "torchvision": _version("torchvision"),\n        "transformers": _version("transformers"),\n        "cuda": torch.version.cuda,\n        "numpy": _version("numpy"),\n        "sam3_commit": sam3_commit() if include_sam3 else None,\n    }\n    gpu = None\n    if torch.cuda.is_available():\n        device = torch.cuda.get_device_properties(0)\n        gpu = {\n            "name": device.name,\n            "compute_capability": [device.major, device.minor],\n            "total_vram_bytes": device.total_memory,\n        }\n    try:\n        result = subprocess.run(\n            ["nvidia-smi"], capture_output=True, text=True, timeout=5, check=False,\n        )\n        nvidia_smi = result.stdout[:8192] if result.returncode == 0 else None\n    except (OSError, subprocess.TimeoutExpired):\n        nvidia_smi = None\n    return {"schema": "autolabel-runtime/v1", "versions": versions,\n            "gpu": gpu, "nvidia_smi": nvidia_smi, "timing_s": {},\n            "peak_cuda_allocated_bytes": {}}\n\n\ndef begin_cuda(torch):\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n\n\ndef end_cuda(torch):\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n        return torch.cuda.max_memory_allocated()\n    return None\n\n\ndef duration_stats(samples):\n    if not samples:\n        return {"count": 0, "mean": None, "p50": None, "p95": None}\n    ordered = sorted(samples)\n\n    def percentile(fraction):\n        index = (len(ordered) - 1) * fraction\n        lower = int(index)\n        upper = min(lower + 1, len(ordered) - 1)\n        return ordered[lower] + (ordered[upper] - ordered[lower]) * (index - lower)\n\n    return {"count": len(samples), "mean": statistics.fmean(samples),\n            "p50": percentile(0.5), "p95": percentile(0.95)}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        while chunk := handle.read(1024 * 1024):\n            digest.update(chunk)\n    return digest.hexdigest()\n', _module.__dict__)
sys.modules['kaggle.runtime_probe'] = _module
_module = types.ModuleType('kaggle.checkpoint_contract')
exec('"""CPU-only checkpoint integrity contract shared by backend and GPU notebook."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nfrom typing import Any\n\n\ndef checkpoint_identity(frames: list[dict[str, Any]], row: dict[str, Any], code_sha256: str = "") -> str:\n    return hashlib.sha256(json.dumps([frames, row, code_sha256], sort_keys=True,\n                                    separators=(",", ":")).encode()).hexdigest()\n\n\ndef model_identity(probe: dict[str, Any]) -> dict[str, Any]:\n    identity = {"sam3_commit": probe.get("environment", {}).get("versions", {}).get("sam3_commit"),\n                "checkpoint_sha256": probe.get("checkpoint_sha256")}\n    if probe.get("model_variant") not in {None, "sam3"}:\n        identity["model_variant"] = probe["model_variant"]\n    return identity\n\n\ndef validate_checkpoint(entry: dict[str, Any], *, identity: str, window_id: int,\n                        frame_ids: list[int], model: dict[str, Any] | None = None) -> None:\n    body = {key: value for key, value in entry.items() if key != "output_sha256"}\n    digest = hashlib.sha256(json.dumps(body, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n    if (entry.get("schema") != "autolabel-sam3-window/v1" or entry.get("input_sha256") != identity\n            or entry.get("window_id") != window_id or entry.get("frame_ids") != frame_ids\n            or entry.get("output_sha256") != digest or (model is not None and entry.get("model") != model)):\n        raise ValueError("window checkpoint provenance or hash mismatch")\n    if not isinstance(entry.get("predictions"), list):\n        raise ValueError("invalid checkpoint predictions")\n    for prediction in entry["predictions"]:\n        if prediction.get("frame_index") not in frame_ids or prediction.get("window_id") != window_id:\n            raise ValueError("checkpoint prediction outside window")\n        box = prediction.get("bbox")\n        if (not isinstance(prediction.get("track_id"), str) or not prediction["track_id"]\n                or not isinstance(prediction.get("label"), str) or not prediction["label"]\n                or not isinstance(box, list) or len(box) != 4\n                or any(type(x) not in {int, float} or not math.isfinite(x) or x < 0 for x in box)\n                or box[0] >= box[2] or box[1] >= box[3]):\n            raise ValueError("invalid checkpoint box or track")\n        raw = prediction.get("mask_rle")\n        if raw:\n            h, w, counts = raw["height"], raw["width"], raw["counts"]\n            digest = hashlib.sha256(json.dumps([h, w, counts], separators=(",", ":")).encode("ascii")).hexdigest()\n            if (type(h) is not int or type(w) is not int or min(h, w) < 1\n                    or not isinstance(counts, list) or any(type(x) is not int or x < 0 for x in counts)\n                    or sum(counts) != h * w or digest != raw["sha256"]):\n                raise ValueError("checkpoint mask checksum mismatch")\n\n', _module.__dict__)
sys.modules['kaggle.checkpoint_contract'] = _module


In [ ]:
main()
